In [ ]:
!pip install datasets transformers

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# A100 optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
dataset = load_dataset(
    "CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary","en",
    split="train",
    streaming=True
)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/415 [00:00<?, ?it/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-base")

# important if pad_token is missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(text, max_len):
    tokens = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    return tokens

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/356 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

# -----------------------------
# RMSNorm
# -----------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return self.scale * x * torch.rsqrt(norm + self.eps)


# -----------------------------
# GELU FFN (renamed from SwiGLU)
# -----------------------------
class GELU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim, bias=False),
            nn.GELU(),
            nn.Linear(hidden_dim, dim, bias=False),
        )

    def forward(self, x):
        return self.net(x)


# -----------------------------
# RoPE
# -----------------------------
class RotaryEmbedding(nn.Module): # Inherit from nn.Module
    def __init__(self, dim):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq) # Register as buffer

    def get_embed(self, seq_len, device):
        # inv_freq is now correctly on the device if the model was moved
        t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
        # No need for .to(device) here, it's already there
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        return torch.cat((freqs, freqs), dim=-1)


def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, freqs_tensor): # Renamed argument to be clearer
    cos = torch.cos(freqs_tensor)[None, None, :, :] # Use torch.cos()
    sin = torch.sin(freqs_tensor)[None, None, :, :] # Use torch.sin()

    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k


# -----------------------------
# Attention (QK Norm + RoPE)
# -----------------------------
class Attention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)

        self.rope = RotaryEmbedding(self.head_dim)

        # Learnable temperature for QK norm (initialized to 15.0)
        self.temperature = nn.Parameter(torch.tensor(15.0))

    def forward(self, x, mask=None):
        B, T, C = x.shape

        qkv = self.qkv(x)
        qkv = qkv.view(B, T, 3, self.heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # RoPE
        rope_emb = self.rope.get_embed(T, x.device)
        q, k = apply_rope(q, k, rope_emb) # Pass the freqs tensor

        # QK normalization
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        # attention with learnable temperature
        attn = (q @ k.transpose(-2, -1)) * self.temperature

        if mask is not None:
            mask = mask[:, None, None, :]
            attn = attn.masked_fill(mask == 0, -1e9)

        attn = attn - attn.max(dim=-1, keepdim=True).values
        attn = torch.softmax(attn, dim=-1)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out(out)


# -----------------------------
# Transformer Block
# -----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, ffn_dim, dropout=0.0):
        super().__init__()

        self.norm1 = RMSNorm(dim)
        self.attn = Attention(dim, heads)

        self.norm2 = RMSNorm(dim)
        self.ffn = GELU(dim, ffn_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


# -----------------------------
# Projection Head
# -----------------------------
class ProjectionHead(nn.Module):
    def __init__(self, dim, proj_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim, bias=False),
            nn.GELU(),
            nn.Linear(dim, proj_dim, bias=False),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


# -----------------------------
# Full Model
# -----------------------------
class RockyEmbeddingModel(nn.Module):
    def __init__(
        self,
        vocab_size=tokenizer.vocab_size,
        dim=768,
        depth=12,
        heads=12,
        ffn_dim=2048,
        proj_dim=512,
        max_seq_len = 1024
    ):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, dim)

        self.layers = nn.ModuleList([
            TransformerBlock(dim, heads, ffn_dim)
            for _ in range(depth)
        ])

        self.norm = RMSNorm(dim)
        self.projection = ProjectionHead(dim, proj_dim)
        self.max_seq_len = max_seq_len

    def forward(self, input_ids, attention_mask=None, return_raw=False):
        if attention_mask is not None:
          attention_mask = attention_mask.long()
        if input_ids.shape[1] > self.max_seq_len:
          raise ValueError(f"Sequence length exceeds max_seq_len={self.max_seq_len}")
        x = self.token_emb(input_ids)

        for layer in self.layers:
            x = layer(x, attention_mask)

        x = self.norm(x)

        # mean pooling
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1)
            x = x * mask
            pooled = x.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        else:
            pooled = x.mean(dim=1)

        if return_raw:
            return pooled

        return self.projection(pooled)


# -----------------------------
# Gradient Safety Utilities
# -----------------------------
def apply_gradient_safeguards(model, max_norm=1.0):
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

    for name, param in model.named_parameters():
        if param.grad is not None:
            if torch.isnan(param.grad).any() or torch.isinf(param.grad).any():
                print(f"[WARNING] Gradient explosion detected in {name}")
                param.grad = torch.nan_to_num(param.grad, nan=0.0, posinf=1.0, neginf=-1.0)


In [ ]:
def unpack_binary(x, dim=1024):
    # ensure uint8 and stay on same device
    x = x.to(dtype=torch.uint8)

    device = x.device

    mask = (2 ** torch.arange(8, device=device, dtype=torch.uint8)).view(1, 1, 8)

    bits = (x.unsqueeze(-1) & mask) != 0
    bits = bits.view(x.shape[0], -1).float()

    bits = bits[:, :dim]

    return bits * 2 - 1

In [ ]:
class StreamingEmbeddingDataset(IterableDataset):
    def __init__(self, dataset, max_len=256):
        self.dataset = dataset
        self.max_len = max_len

    def __iter__(self):
        for item in self.dataset:
            text = item["text"]
            teacher_emb = item["emb_int8"]

            tokens = tokenize(text, self.max_len)

            input_ids = tokens["input_ids"].squeeze(0)
            attention_mask = tokens["attention_mask"].squeeze(0)

            # Yield the raw int8 tensor. Unpacking will happen in the training loop
            # where the batch dimension is present.
            yield {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "emb_int8": torch.tensor(teacher_emb)
            }

In [ ]:
import math

model = RockyEmbeddingModel(
    vocab_size=tokenizer.vocab_size,
    dim=768,
    depth=12,
    heads=12,
    ffn_dim=2048,
    proj_dim=1024   # 🔥 match teacher
)

# ==========================================
# Transformer Weight Initialization
# ==========================================
def init_weights(module):
    if isinstance(module, nn.Linear):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

model.apply(init_weights)

# Scale residual projections to prevent variance explosion
# 12 is the depth of the model
for pn, p in model.named_parameters():
    if pn.endswith('out.weight') or pn.endswith('net.2.weight'):
        torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * 12))

# Move to device after initialization
model = model.to(device)


In [ ]:
def distillation_loss(student, teacher):
    # Normalize both embeddings to unit length
    student = F.normalize(student, dim=-1)
    teacher = F.normalize(teacher, dim=-1)

    # Direct Mean Squared Error between the normalized embeddings
    # This is highly stable and equivalent to minimizing (1 - cosine_similarity)
    return F.mse_loss(student, teacher)


In [ ]:
train_dataset = StreamingEmbeddingDataset(dataset)

dataloader = DataLoader(
    train_dataset,
    batch_size=128,
    num_workers=0, # Changed to 0 to prevent duplicated data streams
    pin_memory=True
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

In [ ]:
# =========================================
# CONFIG
# =========================================
GRAD_NORM_HIGH = 10
GRAD_NORM_LOW = 1e-5  # Lowered to account for MSE mean reduction scale
NEAR_ZERO_THRESH = 1e-7
EMBED_STD_LOW = 0.01
EMBED_STD_HIGH = 10



# =========================================
# GRADIENT HEALTH
# =========================================
def check_gradient_health(model):
    total_norm_sq = 0.0
    max_grad = 0.0
    min_grad = float("inf")
    near_zero = 0
    count = 0

    for p in model.parameters():
        if p.grad is not None:
            g = p.grad.detach()

            norm = g.norm().item()
            total_norm_sq += norm ** 2

            max_grad = max(max_grad, g.abs().max().item())
            min_grad = min(min_grad, g.abs().min().item())

            near_zero += (g.abs() < NEAR_ZERO_THRESH).float().mean().item()
            count += 1

    total_norm = total_norm_sq ** 0.5
    near_zero_ratio = near_zero / max(count, 1)

    print("\n[Gradients]")
    print(f"Global norm: {total_norm:.4f}")
    print(f"Max grad: {max_grad:.6f}")
    print(f"Min grad: {min_grad:.10f}")
    print(f"Near-zero ratio: {near_zero_ratio:.2f}")

    if total_norm > GRAD_NORM_HIGH:
        print("Exploding gradients")
        return "bad"
    elif total_norm < GRAD_NORM_LOW:
        print("Vanishing gradients")
        return "bad"
    elif near_zero_ratio > 0.5:
        print("Many inactive params")
        return "plateau"
    else:
        print("Healthy gradients")
        return "good"


# =========================================
# EMBEDDING HEALTH
# =========================================
def check_embedding_health(embeddings):
    mean = embeddings.mean().item()
    std = embeddings.std().item()

    print("\n[Embeddings]")
    print(f"Mean: {mean:.4f} | Std: {std:.4f}")

    if std < EMBED_STD_LOW:
        print("Embedding collapse")
        return "bad"
    elif std > EMBED_STD_HIGH:
        print("Unstable embeddings")
        return "bad"
    else:
        print("Healthy embeddings")
        return "good"


# =========================================
# MAIN HEALTH CHECK
# =========================================
def model_health_check(model=model, dataloader=dataloader, device=device):
    model.eval()

    batch = next(iter(dataloader))

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    teacher_emb = unpack_binary(batch["emb_int8"].to(device), dim=1024)

    model.zero_grad()

    # Forward
    student = model(input_ids, attention_mask)

    loss = distillation_loss(student, teacher_emb)
    loss.backward()

    print("\n==============================")
    print("  MODEL HEALTH CHECK")
    print("==============================")

    grad_status = check_gradient_health(model)
    emb_status  = check_embedding_health(student)

    model.train()

    # =========================================
    # FINAL DECISION
    # =========================================
    print("\n[Decision]")

    if "bad" in [grad_status, emb_status]:
        print("STOP / FIX TRAINING")
        return "stop"

    elif "plateau" in grad_status:
        print("Plateau detected → consider LR decay or stop soon")
        return "plateau"

    else:
        print("Continue training")
        return "continue"


In [ ]:
from transformers import get_linear_schedule_with_warmup

num_epochs = 1
steps_per_epoch = 50000
total_steps = num_epochs * steps_per_epoch
warmup_steps = 5000

# Create the learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

model.to(device)
model.train()

for epoch in range(num_epochs):
    print(f"\n===== Epoch {epoch+1}/{num_epochs} =====")

    step = 0
    data_iter = iter(dataloader)

    while step < steps_per_epoch:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        teacher_emb = unpack_binary(batch["emb_int8"].to(device), dim=1024)

        optimizer.zero_grad()

        # -----------------------------
        # Forward
        # -----------------------------
        student = model(input_ids, attention_mask)

        # -----------------------------
        # Loss
        # -----------------------------
        loss = distillation_loss(student, teacher_emb)

        # -----------------------------
        # NaN / Inf guard
        # -----------------------------
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"Skipping step {step}")
            continue

        # -----------------------------
        # Backward
        # -----------------------------
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        if grad_norm > 10:
            print(f"[WARNING] High grad norm: {grad_norm:.2f} at step {step}")

        optimizer.step()
        scheduler.step() # <--- Warmup/Decay step

        # -----------------------------
        # Step updates (ONLY HERE)
        # -----------------------------
        step += 1

        # -----------------------------
        # Logging + health check
        # -----------------------------
        if step % 100 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"\nEpoch {epoch+1} | Step {step} | Loss: {loss.item():.4f} | LR: {current_lr:.2e}")

        if step % 5000 == 0 and step != 0:
          torch.save({
              "model": model.state_dict(),
              "optimizer": optimizer.state_dict(),
              "scheduler": scheduler.state_dict(),
              "step": step
          }, "/content/drive/MyDrive/rocky_ckpt.pt")

          print("[CHECKPOINT SAVED] at /content/drive/MyDrive/")


          status = model_health_check(model, dataloader, device)
          model.zero_grad() #  Clear dummy gradients from health check

          if status == "stop":
            print("Stopping training")
            break



===== Epoch 1/1 =====

Epoch 1 | Step 100 | Loss: 0.0017 | LR: 2.00e-06

Epoch 1 | Step 200 | Loss: 0.0016 | LR: 4.00e-06

Epoch 1 | Step 300 | Loss: 0.0016 | LR: 6.00e-06

Epoch 1 | Step 400 | Loss: 0.0016 | LR: 8.00e-06

Epoch 1 | Step 500 | Loss: 0.0016 | LR: 1.00e-05

Epoch 1 | Step 600 | Loss: 0.0016 | LR: 1.20e-05

Epoch 1 | Step 700 | Loss: 0.0016 | LR: 1.40e-05

Epoch 1 | Step 800 | Loss: 0.0016 | LR: 1.60e-05

Epoch 1 | Step 900 | Loss: 0.0016 | LR: 1.80e-05

Epoch 1 | Step 1000 | Loss: 0.0016 | LR: 2.00e-05

Epoch 1 | Step 1100 | Loss: 0.0016 | LR: 2.20e-05

Epoch 1 | Step 1200 | Loss: 0.0016 | LR: 2.40e-05

Epoch 1 | Step 1300 | Loss: 0.0016 | LR: 2.60e-05

Epoch 1 | Step 1400 | Loss: 0.0016 | LR: 2.80e-05

Epoch 1 | Step 1500 | Loss: 0.0016 | LR: 3.00e-05

Epoch 1 | Step 1600 | Loss: 0.0016 | LR: 3.20e-05

Epoch 1 | Step 1700 | Loss: 0.0016 | LR: 3.40e-05

Epoch 1 | Step 1800 | Loss: 0.0016 | LR: 3.60e-05

Epoch 1 | Step 1900 | Loss: 0.0016 | LR: 3.80e-05

Epoch 1 | Step 2

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary/resolve/590b836e34a1f86ad367927e430a1e8732b9d403/en/0010.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary/resolve/590b836e34a1f86ad367927e430a1e8732b9d403/en/0010.parquet
Retrying in 1s [Retry 1/5].



Epoch 1 | Step 7900 | Loss: 0.0015 | LR: 9.36e-05

Epoch 1 | Step 8000 | Loss: 0.0015 | LR: 9.33e-05

Epoch 1 | Step 8100 | Loss: 0.0015 | LR: 9.31e-05

Epoch 1 | Step 8200 | Loss: 0.0015 | LR: 9.29e-05

Epoch 1 | Step 8300 | Loss: 0.0015 | LR: 9.27e-05

Epoch 1 | Step 8400 | Loss: 0.0015 | LR: 9.24e-05

Epoch 1 | Step 8500 | Loss: 0.0015 | LR: 9.22e-05

Epoch 1 | Step 8600 | Loss: 0.0015 | LR: 9.20e-05


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary/resolve/590b836e34a1f86ad367927e430a1e8732b9d403/en/0011.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary/resolve/590b836e34a1f86ad367927e430a1e8732b9d403/en/0011.parquet
Retrying in 1s [Retry 1/5].



Epoch 1 | Step 8700 | Loss: 0.0015 | LR: 9.18e-05

Epoch 1 | Step 8800 | Loss: 0.0016 | LR: 9.16e-05

Epoch 1 | Step 8900 | Loss: 0.0015 | LR: 9.13e-05

Epoch 1 | Step 9000 | Loss: 0.0015 | LR: 9.11e-05

Epoch 1 | Step 9100 | Loss: 0.0015 | LR: 9.09e-05

Epoch 1 | Step 9200 | Loss: 0.0015 | LR: 9.07e-05

Epoch 1 | Step 9300 | Loss: 0.0015 | LR: 9.04e-05

Epoch 1 | Step 9400 | Loss: 0.0015 | LR: 9.02e-05

Epoch 1 | Step 9500 | Loss: 0.0015 | LR: 9.00e-05

Epoch 1 | Step 9600 | Loss: 0.0015 | LR: 8.98e-05

Epoch 1 | Step 9700 | Loss: 0.0015 | LR: 8.96e-05

Epoch 1 | Step 9800 | Loss: 0.0015 | LR: 8.93e-05

Epoch 1 | Step 9900 | Loss: 0.0015 | LR: 8.91e-05

Epoch 1 | Step 10000 | Loss: 0.0015 | LR: 8.89e-05
[CHECKPOINT SAVED] at /content/drive/MyDrive/

  MODEL HEALTH CHECK

[Gradients]
Global norm: 0.0003
Max grad: 0.000006
Min grad: 0.0000000000
Near-zero ratio: 0.99
Many inactive params

[Embeddings]
Mean: -0.0034 | Std: 0.0311
Healthy embeddings

[Decision]
Plateau detected → conside

...


Epoch 1 | Step 12600 | Loss: 0.0015 | LR: 8.31e-05

Epoch 1 | Step 12700 | Loss: 0.0015 | LR: 8.29e-05

Epoch 1 | Step 12800 | Loss: 0.0015 | LR: 8.27e-05

Epoch 1 | Step 12900 | Loss: 0.0015 | LR: 8.24e-05

Epoch 1 | Step 13000 | Loss: 0.0015 | LR: 8.22e-05

Epoch 1 | Step 13100 | Loss: 0.0015 | LR: 8.20e-05

Epoch 1 | Step 13200 | Loss: 0.0015 | LR: 8.18e-05

Epoch 1 | Step 13300 | Loss: 0.0015 | LR: 8.16e-05

Epoch 1 | Step 13400 | Loss: 0.0015 | LR: 8.13e-05

Epoch 1 | Step 13500 | Loss: 0.0015 | LR: 8.11e-05

Epoch 1 | Step 13600 | Loss: 0.0015 | LR: 8.09e-05

Epoch 1 | Step 13700 | Loss: 0.0015 | LR: 8.07e-05

Epoch 1 | Step 13800 | Loss: 0.0015 | LR: 8.04e-05

Epoch 1 | Step 13900 | Loss: 0.0015 | LR: 8.02e-05

Epoch 1 | Step 14000 | Loss: 0.0015 | LR: 8.00e-05

Epoch 1 | Step 14100 | Loss: 0.0015 | LR: 7.98e-05

Epoch 1 | Step 14200 | Loss: 0.0015 | LR: 7.96e-05

Epoch 1 | Step 14300 | Loss: 0.0015 | LR: 7.93e-05

Epoch 1 | Step 14400 | Loss: 0.0015 | LR: 7.91e-05

Epoch 1 | S

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary/resolve/590b836e34a1f86ad367927e430a1e8732b9d403/en/0046.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary/resolve/590b836e34a1f86ad367927e430a1e8732b9d403/en/0046.parquet
Retrying in 2s [Retry 2/5].



Epoch 1 | Step 36000 | Loss: 0.0014 | LR: 3.11e-05


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary/resolve/590b836e34a1f86ad367927e430a1e8732b9d403/en/0046.parquet
Retrying in 1s [Retry 1/5].



Epoch 1 | Step 36100 | Loss: 0.0014 | LR: 3.09e-05

Epoch 1 | Step 36200 | Loss: 0.0014 | LR: 3.07e-05

Epoch 1 | Step 36300 | Loss: 0.0014 | LR: 3.04e-05

Epoch 1 | Step 36400 | Loss: 0.0014 | LR: 3.02e-05

Epoch 1 | Step 36500 | Loss: 0.0014 | LR: 3.00e-05

Epoch 1 | Step 36600 | Loss: 0.0014 | LR: 2.98e-05

Epoch 1 | Step 36700 | Loss: 0.0014 | LR: 2.96e-05

Epoch 1 | Step 36800 | Loss: 0.0014 | LR: 2.93e-05

Epoch 1 | Step 36900 | Loss: 0.0014 | LR: 2.91e-05

Epoch 1 | Step 37000 | Loss: 0.0014 | LR: 2.89e-05

Epoch 1 | Step 37100 | Loss: 0.0014 | LR: 2.87e-05

Epoch 1 | Step 37200 | Loss: 0.0014 | LR: 2.84e-05

Epoch 1 | Step 37300 | Loss: 0.0014 | LR: 2.82e-05

Epoch 1 | Step 37400 | Loss: 0.0014 | LR: 2.80e-05

Epoch 1 | Step 37500 | Loss: 0.0014 | LR: 2.78e-05

Epoch 1 | Step 37600 | Loss: 0.0014 | LR: 2.76e-05

Epoch 1 | Step 37700 | Loss: 0.0014 | LR: 2.73e-05

Epoch 1 | Step 37800 | Loss: 0.0014 | LR: 2.71e-05

Epoch 1 | Step 37900 | Loss: 0.0014 | LR: 2.69e-05

Epoch 1 | S

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/hf_dir/rocky_embed.bin")

In [ ]:
# Testing script
import torch
import torch.nn.functional as F

model.eval()

query = "What is the capital of France?"
options = [
    "Paris is the capital of France.",
    "A completely unrelated sentence about dogs.",
    "France is a country in Western Europe."
]

# Combine for batch processing
sentences = [query] + options

print("Tokenizing sentences...")
inputs = tokenize(sentences, max_len=64)
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

print("Generating embeddings...")
with torch.no_grad():
    embeddings = model(input_ids, attention_mask)

print(f"Embeddings shape: {embeddings.shape}\n")

# Split query and options embeddings
query_emb = embeddings[0].unsqueeze(0)  # [1, D]
option_embs = embeddings[1:]            # [3, D]

# Calculate cosine similarities in one go
similarities = F.cosine_similarity(query_emb, option_embs)

# Rank the options
ranked_results = []
for i, score in enumerate(similarities):
    ranked_results.append((score.item(), options[i]))

ranked_results.sort(key=lambda x: x[0], reverse=True)

print(f"Query: '{query}'\n")
print("Options ranked by similarity:")
for rank, (score, option) in enumerate(ranked_results, 1):
    print(f"{rank}. Score: {score:.4f} | '{option}'")

Tokenizing sentences...
Generating embeddings...
Embeddings shape: torch.Size([4, 1024])

Query: 'What is the capital of France?'

Options ranked by similarity:
1. Score: 0.7481 | 'Paris is the capital of France.'
2. Score: 0.6162 | 'France is a country in Western Europe.'
3. Score: 0.5861 | 'A completely unrelated sentence about dogs.'


In [ ]:
# Get ready to upload the model to huggingface. I'm already authenticated using HF_TOKEN
# Make the model AutoModel ready
# Write script to write neccesary files like modeling_Rocky.py and model_config.json

import os

hf_dir = "/content/rocky_hf"
os.makedirs(hf_dir, exist_ok=True)

# ==========================================
# 1. Write configuration_rocky.py
# ==========================================
config_content = '''from transformers import PretrainedConfig

class RockyConfig(PretrainedConfig):
    model_type = "rocky"

    def __init__(
        self,
        vocab_size=30522,
        dim=768,
        depth=12,
        heads=12,
        ffn_dim=2048,
        proj_dim=1024,
        max_seq_len=1024,
        **kwargs
    ):
        self.vocab_size = vocab_size
        self.dim = dim
        self.depth = depth
        self.heads = heads
        self.ffn_dim = ffn_dim
        self.proj_dim = proj_dim
        self.max_seq_len = max_seq_len
        super().__init__(**kwargs)
        self.auto_map = {
            "AutoConfig": "configuration_rocky.RockyConfig",
            "AutoModel": "modeling_rocky.RockyForEmbeddings"
        }
'''

with open(f"{hf_dir}/configuration_rocky.py", "w") as f:
    f.write(config_content)

# ==========================================
# 2. Write modeling_rocky.py
# ==========================================
modeling_content = '''import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PreTrainedModel
from configuration_rocky import RockyConfig

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return self.scale * x * torch.rsqrt(norm + self.eps)

class GELU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim, bias=False),
            nn.GELU(),
            nn.Linear(hidden_dim, dim, bias=False),
        )

    def forward(self, x):
        return self.net(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def get_embed(self, seq_len, device):
        t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        return torch.cat((freqs, freqs), dim=-1)

def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, freqs_tensor):
    cos = torch.cos(freqs_tensor)[None, None, :, :]
    sin = torch.sin(freqs_tensor)[None, None, :, :]
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class Attention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryEmbedding(self.head_dim)
        self.temperature = nn.Parameter(torch.tensor(15.0))

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x)
        qkv = qkv.view(B, T, 3, self.heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        rope_emb = self.rope.get_embed(T, x.device)
        q, k = apply_rope(q, k, rope_emb)

        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        attn = (q @ k.transpose(-2, -1)) * self.temperature

        if mask is not None:
            mask = mask[:, None, None, :]
            attn = attn.masked_fill(mask == 0, -1e9)

        attn = attn - attn.max(dim=-1, keepdim=True).values
        attn = torch.softmax(attn, dim=-1)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, ffn_dim, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = Attention(dim, heads)
        self.norm2 = RMSNorm(dim)
        self.ffn = GELU(dim, ffn_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x

class ProjectionHead(nn.Module):
    def __init__(self, dim, proj_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim, bias=False),
            nn.GELU(),
            nn.Linear(dim, proj_dim, bias=False),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

class RockyForEmbeddings(PreTrainedModel):
    config_class = RockyConfig

    def __init__(self, config):
        super().__init__(config)
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.dim)

        self.layers = nn.ModuleList([
            TransformerBlock(config.dim, config.heads, config.ffn_dim)
            for _ in range(config.depth)
        ])

        self.norm = RMSNorm(config.dim)
        self.projection = ProjectionHead(config.dim, config.proj_dim)

        self.post_init()

    def forward(self, input_ids, attention_mask=None, return_raw=False):
        if attention_mask is not None:
            attention_mask = attention_mask.long()

        x = self.token_emb(input_ids)

        for layer in self.layers:
            x = layer(x, attention_mask)

        x = self.norm(x)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1)
            x = x * mask
            pooled = x.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        else:
            pooled = x.mean(dim=1)

        if return_raw:
            return pooled

        return self.projection(pooled)
'''

with open(f"{hf_dir}/modeling_rocky.py", "w") as f:
    f.write(modeling_content)

print(f"Successfully generated configuration_rocky.py and modeling_rocky.py in '{hf_dir}' directory.")

Successfully generated configuration_rocky.py and modeling_rocky.py in '/content/rocky_hf' directory.


In [ ]:
import os
import importlib.util
import sys
from transformers import AutoConfig, AutoModel

hf_dir = "/content/rocky_hf"

# Dynamically load configuration_rocky.py
config_file_path = os.path.join(hf_dir, "configuration_rocky.py")
spec_config = importlib.util.spec_from_file_location("configuration_rocky", config_file_path)
configuration_rocky_module = importlib.util.module_from_spec(spec_config)
sys.modules["configuration_rocky"] = configuration_rocky_module
spec_config.loader.exec_module(configuration_rocky_module)
RockyConfig = configuration_rocky_module.RockyConfig

# Dynamically load modeling_rocky.py
modeling_file_path = os.path.join(hf_dir, "modeling_rocky.py")
spec_modeling = importlib.util.spec_from_file_location("modeling_rocky", modeling_file_path)
modeling_rocky_module = importlib.util.module_from_spec(spec_modeling)
sys.modules["modeling_rocky"] = modeling_rocky_module
spec_modeling.loader.exec_module(modeling_rocky_module)
RockyForEmbeddings = modeling_rocky_module.RockyForEmbeddings

# 1. Register the custom configurations locally
AutoConfig.register("rocky", RockyConfig)
AutoModel.register(RockyConfig, RockyForEmbeddings)

# 2. Register the custom classes for AutoModel compatibility during push
RockyConfig.register_for_auto_class()
RockyForEmbeddings.register_for_auto_class("AutoModel")

# 3. Instantiate the custom config and model
hf_config = RockyConfig(vocab_size=tokenizer.vocab_size)
hf_model = RockyForEmbeddings(hf_config)

# 4. Push config, model, and code to Hugging Face Hub
repo_id = "pranavupadhyaya52/rocky-embed"

print(f"Pushing config, custom code, and dummy weights to {repo_id}...")
# Pushing the model uploads the custom python files (modeling/configuration)
hf_config.push_to_hub(repo_id)
hf_model.push_to_hub(repo_id)

print(f"\nSuccess! Settings and custom code updated on https://huggingface.co/{repo_id}")

Pushing config, custom code, and dummy weights to pranavupadhyaya52/rocky-embed...


No files have been modified since last commit. Skipping to prevent empty commit.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t2s2rcb/model.safetensors:   1%|1         | 4.01MB /  364MB            


Success! Settings and custom code updated on https://huggingface.co/pranavupadhyaya52/rocky-embed


---language:- en
tags:- feature-extraction
- sentence-similarity
- custom-code
- knowledge-distillation
pipeline_tag: feature-extraction
library_name: transformers
---

# Model Card: Rocky-Embed

## Model Description
`rocky-embed` is a custom, lightweight Transformer-based text embedding model. It was trained via knowledge distillation using the `CohereLabs/wikipedia-2023-11-embed-multilingual-v3-int8-binary` dataset as a teacher. The model maps sentences and paragraphs to a 1024-dimensional dense vector space and can be used for tasks like clustering or semantic search.

### Architecture Highlights:
* **Custom Transformer Blocks:** Uses RMSNorm for layer normalization and GELU activations.
* **Positional Embeddings:** Implements Rotary Positional Embeddings (RoPE).
* **Attention:** Uses QK Normalization with a learnable temperature parameter.
* **Parameters:**
  * Dimensions: 768
  * Depth: 12 layers
  * Heads: 12
  * Projection Dimension: 1024 (matching the teacher model)

## Training Details
* **Dataset:** Trained on English Wikipedia snippets.
* **Objective:** Direct Mean Squared Error (MSE) distillation from the normalized embeddings of the teacher model.
* **Optimizer:** AdamW with linear learning rate decay and warmup.

## Evaluation Results (STSb)
* **Spearman Correlation:** 0.5453

## How to Use

You can load this model directly from the Hugging Face Hub using the `transformers` library. Since this model uses a custom architecture (`RockyForEmbeddings`), you must pass `trust_remote_code=True` when loading it.

```python
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# 1. Load the tokenizer and model
model_id = "pranavupadhyaya52/rocky-embed"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Important: Set trust_remote_code=True to use the custom Rocky architecture
model = AutoModel.from_pretrained(model_id, trust_remote_code=True)

model.eval()

# 2. Prepare your input texts
queries = [
    "What is the capital of France?",
    "Paris is the capital of France.",
    "A completely unrelated sentence about dogs."
]

# 3. Tokenize
inputs = tokenizer(
    queries,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

# 4. Generate Embeddings
with torch.no_grad():
    # The model outputs the normalized pooled embeddings directly
    embeddings = model(inputs["input_ids"], inputs["attention_mask"])

print("Embeddings shape:", embeddings.shape)

# 5. Compute cosine similarities
query_emb = embeddings[0].unsqueeze(0)
option_embs = embeddings[1:]
similarities = F.cosine_similarity(query_emb, option_embs)

print(f"\nSimilarity with '{queries[1]}': {similarities[0]:.4f}")
print(f"Similarity with '{queries[2]}': {similarities[1]:.4f}")
```

In [ ]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
from scipy.stats import spearmanr
from tqdm import tqdm

# 1. Load the STS Benchmark dataset from GLUE
print("Loading STS Benchmark dataset...")
stsb_val = load_dataset("glue", "stsb", split="validation")

sentences1 = stsb_val["sentence1"]
sentences2 = stsb_val["sentence2"]
true_scores = stsb_val["label"]  # Scores from 0.0 to 5.0

# 2. Put the existing model in evaluation mode
model.eval()

# 3. Helper function to encode a list of sentences in batches
def get_embeddings(sentences, batch_size=64):
    all_embeddings = []
    for i in tqdm(range(0, len(sentences), batch_size), desc="Encoding sentences"):
        batch_sentences = sentences[i : i + batch_size]

        # Tokenize using the already loaded tokenizer
        inputs = tokenizer(
            batch_sentences,
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        # Generate embeddings
        with torch.no_grad():
            embeddings = model(input_ids, attention_mask)

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)

# 4. Encode the sentence pairs
print("\nEncoding first sentences...")
embs1 = get_embeddings(sentences1)

print("\nEncoding second sentences...")
embs2 = get_embeddings(sentences2)

# 5. Compute Cosine Similarities
print("\nComputing cosine similarities...")
cos_sims = F.cosine_similarity(embs1, embs2).numpy()

# 6. Calculate Spearman Correlation
correlation, p_value = spearmanr(cos_sims, true_scores)

print("\n==============================")
print("  EVALUATION RESULTS (STSb)")
print("==============================")
print(f"Number of sentence pairs: {len(true_scores)}")
print(f"Spearman Correlation: {correlation:.4f}")


Loading STS Benchmark dataset...

Encoding first sentences...


Encoding sentences: 100%|██████████| 24/24 [00:01<00:00, 12.48it/s]



Encoding second sentences...


Encoding sentences: 100%|██████████| 24/24 [00:01<00:00, 21.60it/s]


Computing cosine similarities...

  EVALUATION RESULTS (STSb)
Number of sentence pairs: 1500
Spearman Correlation: 0.5453
